In [3]:
# Task 1: LangChain Setup & Core Concepts (Groq)
# pip install langchain langchain-groq langchain-core
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


load_dotenv()
# Setup Groq LLM
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=1024)

# LCEL chain: prompt | llm | parser
prompt = ChatPromptTemplate.from_template("Answer in 2-3 sentences: {question}")
parser = StrOutputParser()
chain = prompt | llm | parser

# Test invoke
result = chain.invoke({"question": "What is LangChain Expression Language?"})
print("INVOKE RESULT:\n", result)

# Test stream
print("\nSTREAM RESULT:")
for chunk in chain.stream({"question": "Why use pipe syntax?"}):
    print(chunk, end="", flush=True)
print()

INVOKE RESULT:
 LangChain Expression Language (LCEL) is a concise, declarative syntax for building LangChain pipelines that lets developers chain together prompts, LLM calls, tools, and other components without writing extensive boilerplate code. By expressing workflows as readable expressions, LCEL simplifies the creation, composition, and debugging of complex LLM‑driven applications.

STREAM RESULT:
Pipe syntax lets you chain operations in a clear, left‑to‑right flow, making code easier to read and reason about because each step receives the output of the previous one. It also encourages composability, letting you build complex behavior from simple, reusable functions without deeply nested calls or temporary variables. In many environments (e.g., Unix shells, functional languages) the pipe automatically handles data passing, reducing boilerplate and potential errors.


In [ ]:
# Task 2: Define & Register Tools
import json
from langchain_core.tools import tool

# Create products.json for the real data source tool
products_data = {"products": [
    {"name": "laptop", "price": 999, "category": "electronics", "stock": 15},
    {"name": "phone", "price": 699, "category": "electronics", "stock": 30},
    {"name": "tablet", "price": 449, "category": "electronics", "stock": 20},
    {"name": "headphones", "price": 149, "category": "accessories", "stock": 50},
    {"name": "smartwatch", "price": 299, "category": "wearables", "stock": 25}
]}
with open("products.json", "w") as f:
    json.dump(products_data, f, indent=2)

# Tool 1: Calculator (reused from Day 1)
@tool
def calculator(operation: str, a: float, b: float) -> str:
    """Performs basic arithmetic (add, subtract) on two numbers. Use this when the user asks for math calculations like adding, subtracting, or any arithmetic operation. Do not use for comparisons or statistics."""
    if operation == "add":
        return f"{a} + {b} = {a + b}"
    elif operation == "subtract":
        return f"{a} - {b} = {a - b}"
    return f"Error: Unknown operation '{operation}'"

# Tool 2: Weather (reused from Day 1)
@tool
def get_weather(city: str) -> str:
    """Returns current temperature for a given city. Use this when the user asks about weather, temperature, or climate conditions for a specific city. Returns fake stub data for testing."""
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15, "berlin": 20}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}°C"

# Tool 3: Product Lookup (NEW - reads real JSON data)
@tool
def product_lookup(product_name: str) -> str:
    """Looks up product information including price, category, and stock availability from the local product database. Use this when the user asks about product prices, availability, or wants to compare products."""
    with open("products.json", "r") as f:
        data = json.load(f)
    for p in data["products"]:
        if product_name.lower() in p["name"].lower():
            return f"Product: {p['name']}, Price: ${p['price']}, Category: {p['category']}, Stock: {p['stock']} units"
    return f"Product '{product_name}' not found in database"

# Register tools
tools = [calculator, get_weather, product_lookup]
# Test each tool individually
print("=== TOOL TESTS ===")
print("calculator:", calculator.invoke({"operation": "add", "a": 10, "b": 5}))
print("get_weather:", get_weather.invoke({"city": "Tokyo"}))
print("product_lookup:", product_lookup.invoke({"product_name": "laptop"}))

# Show tool schemas (what the LLM sees)
print("\n=== TOOL SCHEMAS ===")
for t in tools:
    print(f"\n{t.name}: {t.description}")
    print(f"  Schema: {t.args_schema.model_json_schema()}")

=== TOOL TESTS ===
calculator: 10.0 + 5.0 = 15.0
get_weather: Weather in Tokyo: 22°C
product_lookup: Product: laptop, Price: $999, Category: electronics, Stock: 15 units

=== TOOL SCHEMAS ===

calculator: Performs basic arithmetic (add, subtract) on two numbers. Use this when the user asks for math calculations like adding, subtracting, or any arithmetic operation. Do not use for comparisons or statistics.
  Schema: {'description': 'Performs basic arithmetic (add, subtract) on two numbers. Use this when the user asks for math calculations like adding, subtracting, or any arithmetic operation. Do not use for comparisons or statistics.', 'properties': {'operation': {'title': 'Operation', 'type': 'string'}, 'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['operation', 'a', 'b'], 'title': 'calculator', 'type': 'object'}

get_weather: Returns current temperature for a given city. Use this when the user asks about weather, temperature, or climate co

In [16]:
# Task 3: Build an Agent (langchain 1.3.9)
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
import json

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=1024)

@tool
def calculator(operation: str, a: float, b: float) -> str:
    """Performs basic arithmetic (add, subtract) on two numbers. Use this when the user asks for math calculations like adding, subtracting, or any arithmetic operation. Do not use for comparisons or statistics."""
    if operation == "add":
        return f"{a} + {b} = {a + b}"
    elif operation == "subtract":
        return f"{a} - {b} = {a - b}"
    return f"Error: Unknown operation '{operation}'"

@tool
def get_weather(city: str) -> str:
    """Returns current temperature for a given city. Use this when the user asks about weather, temperature, or climate conditions for a specific city. Returns fake stub data for testing."""
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15, "berlin": 20}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}°C"

@tool
def product_lookup(product_name: str) -> str:
    """Looks up product information including price, category, and stock availability from the local product database. Use this when the user asks about product prices, availability, or wants to compare products."""
    with open("products.json", "r") as f:
        data = json.load(f)
    for p in data["products"]:
        if product_name.lower() in p["name"].lower():
            return f"Product: {p['name']}, Price: ${p['price']}, Category: {p['category']}, Stock: {p['stock']} units"
    return f"Product '{product_name}' not found in database"

tools = [calculator, get_weather, product_lookup]

# Create agent with debug=True to see reasoning trace
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use the tools available to answer questions.",
    debug=True,
)

# Test 1: Multi-step weather comparison
print("=" * 60)
print("TEST 1: Multi-step weather comparison")
print("=" * 60)
result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Tokyo and Paris? Which is warmer?"}]})
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

# Test 2: Product lookup
print("\n" + "=" * 60)
print("TEST 2: Product lookup")
print("=" * 60)
result = agent.invoke({"messages": [{"role": "user", "content": "How much does the laptop cost?"}]})
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

# Test 3: Combined tools
print("\n" + "=" * 60)
print("TEST 3: Combined calculator + weather")
print("=" * 60)
result = agent.invoke({"messages": [{"role": "user", "content": "What's 25 + 17 and what's the weather in Berlin?"}]})
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

TEST 1: Multi-step weather comparison
[values] {'messages': [HumanMessage(content="What's the weather in Tokyo and Paris? Which is warmer?", additional_kwargs={}, response_metadata={}, id='4cbf69db-e805-4d9d-80f1-a5c76e901b51')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks for weather in Tokyo and Paris, then which is warmer. We need to get weather for both cities using get_weather function. Then compare temperatures. Use function calls. Then compute which is warmer. Use calculator? Actually we need to compare numbers, not arithmetic. Could just do logic in code. But we can retrieve temperatures, then compare manually in response. Use get_weather twice.', 'tool_calls': [{'id': 'fc_1da3783e-2268-4ece-a993-94540da5bb0e', 'function': {'arguments': '{"city":"Tokyo"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 290, 'total_tokens': 397, 'comple

In [ ]:
# Task 4: Add Memory
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
import json

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=1024)

@tool
def calculator(operation: str, a: float, b: float) -> str:
    """Performs basic arithmetic (add, subtract) on two numbers. Use this when the user asks for math calculations like adding, subtracting, or any arithmetic operation. Do not use for comparisons or statistics."""
    if operation == "add":
        return f"{a} + {b} = {a + b}"
    elif operation == "subtract":
        return f"{a} - {b} = {a - b}"
    return f"Error: Unknown operation '{operation}'"

@tool
def get_weather(city: str) -> str:
    """Returns current temperature for a given city. Use this when the user asks about weather, temperature, or climate conditions for a specific city. Returns fake stub data for testing."""
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15, "berlin": 20}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}°C"

@tool
def product_lookup(product_name: str) -> str:
    """Looks up product information including price, category, and stock availability from the local product database. Use this when the user asks about product prices, availability, or wants to compare products."""
    with open("products.json", "r") as f:
        data = json.load(f)
    for p in data["products"]:
        if product_name.lower() in p["name"].lower():
            return f"Product: {p['name']}, Price: ${p['price']}, Category: {p['category']}, Stock: {p['stock']} units"
    return f"Product '{product_name}' not found in database"

tools = [calculator, get_weather, product_lookup]

# Add memory
memory = MemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use the tools available to answer questions.",
    checkpointer=memory,
    debug=True,
)

# Same thread_id = agent remembers previous turns
config = {"configurable": {"thread_id": "thread-1"}}

# Turn 1: Find laptop price
print("=" * 60)
print("TURN 1: Find the price of laptop")
print("=" * 60)
result = agent.invoke({"messages": [{"role": "user", "content": "Find the price of laptop"}]}, config)
for msg in result["messages"]:
    if hasattr(msg, "content") and msg.content:
        print(f"{msg.type}: {msg.content}")

# Turn 2: Compare to phone (agent remembers laptop price from turn 1)
print("\n" + "=" * 60)
print("TURN 2: Now compare it to phone")
print("=" * 60)
result = agent.invoke({"messages": [{"role": "user", "content": "Now compare it to phone"}]}, config)
for msg in result["messages"]:
    if hasattr(msg, "content") and msg.content:
        print(f"{msg.type}: {msg.content}")

# Turn 3: Budget recommendation (agent remembers both from previous turns)
print("\n" + "=" * 60)
print("TURN 3: Which one should I recommend to a budget-conscious client?")
print("=" * 60)
result = agent.invoke({"messages": [{"role": "user", "content": "Which one should I recommend to a budget-conscious client?"}]}, config)
for msg in result["messages"]:
    if hasattr(msg, "content") and msg.content:
        print(f"{msg.type}: {msg.content}")